# Unidad 4 · Cuaderno 03 · Tablas, gráficas e informes

**Modelación y Simulación Computacional** · Maestría en Ingeniería, Universidad de Sucre, periodo 2026-2

**Unidad 4.** Validación, interpretación y comunicación de resultados
· **Subtema del plan 4.3**

Este cuaderno ejecuta lo que el libro expone en la sección 4.5. El texto no
repite la teoría, remite a ella por número de definición, de teorema, de
ejemplo, de listado, de tabla o de figura, y se ocupa de reproducir los
resultados publicados y de verificarlos.

**Autor.** Prof. Daniel Otero Meza, Ing., Ph.D.

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad4/U4_03_tablas_graficas_e_informes.ipynb)

## Objetivos de aprendizaje

1. Decidir entre una tabla y una figura con el criterio de función que la sección 4.5 del libro establece.
2. Generar la tabla de resultados directamente desde un DataFrame, sin transcribir ninguna cifra a mano, según el Listado 4.11.
3. Redondear cada columna con las cifras significativas que la incertidumbre gobierna, de manera automática.
4. Exportar la misma tabla a LaTeX y a Markdown desde una sola fuente, con encabezados que declaren la unidad una sola vez.
5. Producir una figura de informe con la incertidumbre representada y exportarla en PDF vectorial verificable.

## Puesta a punto

In [ ]:
# Puesta a punto. Detecta el entorno e instala solo lo que falte.
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict[str, str]) -> None:
    """Instala los paquetes cuyo módulo no se encuentre en el entorno."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("Entorno listo. Colab:", EN_COLAB)

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

PALETA = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
           "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 10, "axes.grid": True, "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=list(PALETA.values())),
})

# Bandera de los ejercicios guiados. En la versión de trabajo vale False
# para que el cuaderno corra completo aunque falten celdas por resolver.
REVISAR = False


def verificar(nombre: str, obtenido, esperado: float,
              tol: float = 1.0e-3) -> bool:
    """Compara un resultado con el valor esperado sin detener el cuaderno."""
    if obtenido is None or (isinstance(obtenido, float) and np.isnan(obtenido)):
        print(f"[pendiente] {nombre}, la celda marcada COMPLETE sigue sin resolver")
        return False
    escala = abs(esperado) if esperado != 0.0 else 1.0
    error = abs(float(obtenido) - esperado) / escala
    estado = "ok" if error <= tol else "revisar"
    print(f"[{estado}] {nombre}, obtenido {float(obtenido):.6g}, "
          f"esperado {esperado:.6g}, error relativo {error:.2e}")
    if REVISAR:
        assert error <= tol, f"{nombre} no coincide con el valor esperado"
    return error <= tol


print("Semilla del curso:", SEMILLA)

In [ ]:
# Acceso a datos/ que funciona en Colab y en local, sin rutas absolutas.
# Si la carpeta no viaja con el cuaderno, las series se reconstruyen con la
# semilla del curso y con las cifras que el libro publica.

def carpeta_datos() -> Path:
    """Ubica datos/ subiendo por el árbol, o la crea junto al cuaderno."""
    base = Path.cwd()
    for nivel in [base, *base.parents][:4]:
        for candidata in (nivel / "datos", nivel / "03_cuadernos" / "datos"):
            if candidata.is_dir():
                return candidata
    destino = base / "datos"
    destino.mkdir(parents=True, exist_ok=True)
    return destino


def _cinetica_monod() -> pd.DataFrame:
    return pd.DataFrame({
        "S_g_L": [0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 18.0, 25.0, 35.0],
        "mu_1_h": [0.0702, 0.1248, 0.1919, 0.2496, 0.2761,
                   0.3205, 0.3689, 0.3709, 0.3770, 0.3773]})


def _secado_calibracion() -> pd.DataFrame:
    t = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0,
                  4.0, 5.0, 6.0, 7.0, 8.0])
    gen = np.random.default_rng(SEMILLA)
    mr = np.round(np.exp(-0.350 * t**1.15) + gen.normal(0.0, 0.008, t.size), 4)
    return pd.DataFrame({"t_h": t, "MR": mr})


def _secado_validacion() -> pd.DataFrame:
    t = np.array([0.5, 1.0, 1.5, 2.0, 2.75, 3.5, 4.5, 5.5, 6.5, 8.0])
    gen = np.random.default_rng(SEMILLA + 1)
    mr = np.round(np.exp(-0.362 * t**1.15) + gen.normal(0.0, 0.008, t.size), 4)
    return pd.DataFrame({"t_h": t, "MR": mr})


def _caudal_mensual() -> pd.DataFrame:
    obs = [6.21, 5.01, 4.30, 9.39, 14.85, 18.28, 16.85, 17.12, 21.51, 22.75,
           19.33, 12.13, 7.65, 6.14, 4.54, 8.58, 14.73, 21.82, 14.36, 17.57,
           24.54, 31.26, 21.30, 11.27, 9.71, 6.09, 5.39, 10.86, 23.94, 20.68,
           19.39, 22.16, 32.76, 38.82, 21.33, 12.99, 5.97, 5.83, 5.07, 8.20,
           18.23, 18.93, 12.72, 16.63, 22.38, 27.76, 18.00, 12.27]
    sim = [7.70, 4.70, 4.43, 7.83, 14.55, 16.00, 15.33, 17.97, 20.37, 24.45,
           17.19, 13.27, 10.40, 6.93, 5.40, 10.19, 16.72, 22.35, 13.78, 17.44,
           24.20, 29.67, 17.56, 13.20, 12.94, 2.59, 5.60, 12.74, 21.94, 16.05,
           18.74, 17.99, 27.76, 24.48, 15.41, 12.64, 6.83, 6.11, 5.24, 11.99,
           19.88, 20.51, 12.28, 17.89, 19.38, 19.28, 14.48, 8.00]
    return pd.DataFrame({"mes": np.arange(1, 49),
                         "periodo": ["calibracion"] * 24 + ["validacion"] * 24,
                         "Q_obs_m3_s": obs, "Q_sim_m3_s": sim})


def _arreglo_fotovoltaico() -> pd.DataFrame:
    return pd.DataFrame({"configuracion": ["Base", "Optima", "Sombreado"],
                         "H_kWh_m2": [1980.0, 2035.0, 1910.0],
                         "u_rel_H": [0.04, 0.04, 0.04]})


def _entradas_vertedero() -> pd.DataFrame:
    return pd.DataFrame({"magnitud": ["C_d", "b", "h"],
                         "unidad": ["1", "m", "m"],
                         "valor": [0.620, 0.500, 0.150],
                         "u_tipica": [0.015, 0.0010, 0.0015]})


CONSTRUCTORES = {
    "cinetica_monod.csv": _cinetica_monod,
    "secado_maiz_calibracion.csv": _secado_calibracion,
    "secado_maiz_validacion.csv": _secado_validacion,
    "caudal_mensual.csv": _caudal_mensual,
    "arreglo_fotovoltaico.csv": _arreglo_fotovoltaico,
    "entradas_vertedero.csv": _entradas_vertedero,
}

CARPETA_DATOS = carpeta_datos()


def leer_datos(nombre: str) -> pd.DataFrame:
    """Lee un archivo de datos/ y lo reconstruye si no está presente."""
    ruta = CARPETA_DATOS / nombre
    if not ruta.exists():
        CONSTRUCTORES[nombre]().to_csv(ruta, index=False)
    return pd.read_csv(ruta)


print("Carpeta de datos:", CARPETA_DATOS.name)

In [ ]:
# Carpeta de salidas del informe, relativa al cuaderno.
SALIDAS = Path("salidas")
(SALIDAS / "tablas").mkdir(parents=True, exist_ok=True)
(SALIDAS / "figuras").mkdir(parents=True, exist_ok=True)
print("Las tablas y las figuras se escriben en", SALIDAS)

## 1. La tabla y la figura argumentan, no ilustran

La sección 4.5 del libro fija el criterio de función. La tabla sirve
cuando el lector necesita leer valores concretos para compararlos o
reutilizarlos, y la figura cuando lo que importa es una tendencia o
la presencia de estructura. La Figura 4.10 señala cada elemento de una
y de otra.

En la tabla el título va encima y es autosuficiente, la unidad se
declara una sola vez en el encabezado, las cifras significativas las
gobierna la incertidumbre, las reglas son solo horizontales y una
nota al pie fija el alcance. En la figura el pie va debajo e
interpreta en lugar de describir, los ejes llevan símbolo, nombre y
unidad, la incertidumbre se dibuja y la leyenda no repite el pie.

La consecuencia operativa es que ni la tabla ni la figura deben
construirse a mano. Copiar números de una consola a un procesador de
texto introduce errores que nadie detecta y obliga a repetir el
trabajo cada vez que cambia un dato.

## 2. El Ejemplo 4.7, tres configuraciones fotovoltaicas

Un estudio compara tres configuraciones de un arreglo fotovoltaico de
5.00 kW con incertidumbre de 0.10 kW en Sincelejo. La irradiación
anual en el plano del arreglo tiene incertidumbre relativa del 4 por
ciento y la razón de desempeño vale 0.78 con incertidumbre de 0.03.
El Listado 4.11 del libro genera la tabla directamente desde los
datos del proyecto.

In [ ]:
arreglo = leer_datos("arreglo_fotovoltaico.csv")

POTENCIA, U_POTENCIA = 5.00, 0.10       # kW
RAZON_DESEMPENO, U_RAZON = 0.78, 0.03   # adimensional

resultados = arreglo.set_index("configuracion")
u_relativa = np.sqrt((U_POTENCIA / POTENCIA)**2
                     + resultados["u_rel_H"]**2
                     + (U_RAZON / RAZON_DESEMPENO)**2)
resultados["E_kWh"] = POTENCIA * resultados["H_kWh_m2"] * RAZON_DESEMPENO
resultados["u_E_kWh"] = u_relativa * resultados["E_kWh"]
resultados["Y_kWh_kW"] = resultados["E_kWh"] / POTENCIA

print(resultados.to_string(float_format=lambda v: f"{v:.4f}"))
print(f"\nIncertidumbre relativa combinada "
      f"{100 * u_relativa.iloc[0]:.2f} por ciento   (libro 5.90)")
print("Energías anuales en MWh:",
      [f"{v / 1000:.2f}" for v in resultados["E_kWh"]],
      "  (libro 7.72, 7.94 y 7.45)")
print("Incertidumbres en MWh:",
      [f"{v / 1000:.2f}" for v in resultados["u_E_kWh"]],
      "  (libro 0.46, 0.47 y 0.44)")
print("Rendimientos específicos en kWh/kW:",
      [f"{v:.0f}" for v in resultados["Y_kWh_kW"]],
      "  (libro 1544, 1587 y 1490)")

assert abs(100 * u_relativa.iloc[0] - 5.90) < 5.0e-2
assert np.allclose(resultados["E_kWh"] / 1000, [7.72, 7.94, 7.45], atol=5.0e-3)
assert np.allclose(resultados["u_E_kWh"] / 1000, [0.46, 0.47, 0.44], atol=5.0e-3)
assert np.allclose(resultados["Y_kWh_kW"], [1544, 1587, 1490], atol=0.5)

## 3. Cifras significativas gobernadas por la incertidumbre

La sección 4.4.2 del libro fija la regla, la incertidumbre se
redondea a una o dos cifras significativas y el valor a la misma
posición decimal. Automatizar ese redondeo en el código que produce
las tablas evita que un solo número del informe la incumpla.

La función siguiente recibe el DataFrame y la pareja de columnas que
forman valor e incertidumbre, y devuelve las dos ya redondeadas de
manera coherente, cada una con su propio número de decimales.

In [ ]:
def posicion_significativa(u: float, cifras: int = 2) -> int:
    """Posición decimal a la que redondear, positiva hacia la derecha."""
    if not np.isfinite(u) or u == 0.0:
        return 0
    return int(cifras - 1 - np.floor(np.log10(abs(u))))


def decimales_comunes(incertidumbres, cifras: int = 2) -> int:
    """Decimales que hacen coherente toda una columna de la tabla."""
    posiciones = [posicion_significativa(u, cifras) for u in incertidumbres]
    return max(max(posiciones), 0)


def columna_coherente(valores, incertidumbres, cifras: int = 2):
    """Redondea valor e incertidumbre a la misma posición decimal."""
    d = decimales_comunes(incertidumbres, cifras)
    return np.round(valores, d), np.round(incertidumbres, d), d


energia_mwh = resultados["E_kWh"].to_numpy() / 1000.0
u_mwh = resultados["u_E_kWh"].to_numpy() / 1000.0
e_red, u_red, decimales_e = columna_coherente(energia_mwh, u_mwh)

print(f"La incertidumbre pide {decimales_e} decimales en esta columna.")
for nombre, v, u in zip(resultados.index, e_red, u_red):
    print(f"{nombre:10s} {v:.{decimales_e}f} +- {u:.{decimales_e}f} MWh")
assert decimales_e == 2
print("\nEscribir 7.7220 MWh con incertidumbre de 0.46 MWh sería una")
print("contradicción interna, y el redondeo automático la impide.")

## 4. De un DataFrame a LaTeX y a Markdown

La tabla del informe se arma una sola vez y se exporta a los dos
formatos que el curso usa, el de LaTeX para el documento final y el
de Markdown para el repositorio y para la revisión rápida. Ambas
salidas provienen del mismo objeto, de modo que no pueden discrepar.

La unidad se declara una sola vez en el encabezado, según pide la
Figura 4.10, y el redondeo lo gobierna la incertidumbre de la sección
anterior.

In [ ]:
def tabla_de_informe(marco: pd.DataFrame,
                     columnas: dict[str, tuple[str, int]]) -> pd.DataFrame:
    """Construye la tabla ya formateada, con la unidad en el encabezado.

    Cada entrada de ``columnas`` asocia el nombre de la columna con
    el encabezado que llevará y el número de decimales.
    """
    salida = pd.DataFrame(index=marco.index)
    for origen, (encabezado_col, decimales) in columnas.items():
        salida[encabezado_col] = [f"{v:.{decimales}f}" for v in marco[origen]]
    return salida


tabla = pd.DataFrame({
    "H": resultados["H_kWh_m2"],
    "E": e_red,
    "u_E": u_red,
    "Y": resultados["Y_kWh_kW"]})
tabla.index.name = "Configuración"

TABLA_INFORME = tabla_de_informe(tabla, {
    "H": ("H (kWh/m2)", 0),
    "E": ("E (MWh)", decimales_e),
    "u_E": ("u(E) (MWh)", decimales_e),
    "Y": ("Y (kWh/kW)", 0)})
print(TABLA_INFORME.to_string())

### 4.1 Exportación a LaTeX

El método de pandas produce el entorno tabular. El cuaderno le añade
el título encima, las reglas horizontales del paquete de tablas del
libro y la nota al pie que fija el alcance, de modo que la salida se
pueda incluir tal cual en el documento.

In [ ]:
def a_latex(marco: pd.DataFrame, titulo: str, etiqueta: str,
            notas: list[str]) -> str:
    # Tabla de informe completa, con el título encima, las reglas
    # horizontales del paquete booktabs y las notas al pie.
    cuerpo = marco.to_latex(index=True, escape=False,
                            column_format="l" + "r" * marco.shape[1])
    # pandas deja el nombre del índice en una fila propia con celdas
    # vacías. Aquí se traslada al encabezado y esa fila se elimina.
    nombre = str(marco.index.name or "")

    def fila_vacia(linea: str) -> bool:
        celdas = linea.rstrip().removesuffix(r"\\").split("&")
        return (len(celdas) == marco.shape[1] + 1
                and celdas[0].strip() == nombre
                and all(c.strip() == "" for c in celdas[1:]))

    lineas = []
    for linea in cuerpo.splitlines():
        if nombre and fila_vacia(linea):
            continue
        if linea.lstrip().startswith("&"):
            linea = nombre + " " + linea.lstrip()
        lineas.append(linea)
    cuerpo = "\n".join(lineas)
    pie = "\n".join(f"      \\item[{chr(97 + i)}] {t}"
                    for i, t in enumerate(notas))
    return "\n".join([
        r"\begin{table}[htb!]",
        r"  \centering",
        r"  \begin{threeparttable}",
        rf"    \caption{{{titulo}}}",
        rf"    \label{{{etiqueta}}}",
        cuerpo,
        r"    \begin{tablenotes}[flushleft]\footnotesize",
        pie,
        r"    \end{tablenotes}",
        r"  \end{threeparttable}",
        r"\end{table}",
        ""])


NOTAS = ["Energía anual estimada como el producto de la potencia "
         "nominal, la irradiación en el plano y la razón de desempeño.",
         "La incertidumbre relativa combinada vale "
         f"{100 * u_relativa.iloc[0]:.2f} por ciento y es la misma "
         "para las tres configuraciones."]

latex = a_latex(TABLA_INFORME,
                "Energía anual estimada de tres configuraciones del "
                "arreglo fotovoltaico.",
                "tab::arregloFotovoltaico", NOTAS)
(SALIDAS / "tablas" / "arreglo_fotovoltaico.tex").write_text(
    latex, encoding="utf-8")
print(latex)
assert r"\toprule" in latex and r"\caption" in latex
assert r"\begin{threeparttable}" in latex

### 4.2 Exportación a Markdown

El curso no depende de bibliotecas ajenas a las cinco declaradas, de
modo que la conversión a Markdown se implementa aquí. La ventaja de
escribirla es que el ancho de cada columna y la alineación quedan
bajo control, y que la misma función sirve para cualquier tabla del
informe.

In [ ]:
def a_markdown(marco: pd.DataFrame, titulo: str = "",
               notas: list[str] | None = None) -> str:
    """Tabla en Markdown con columnas alineadas, sin dependencias extra."""
    encabezados = [str(marco.index.name or "")] + [str(c) for c in marco.columns]
    filas = [[str(idx)] + [str(v) for v in fila]
             for idx, fila in zip(marco.index, marco.to_numpy())]
    anchos = [max(len(encabezados[j]), *(len(f[j]) for f in filas))
              for j in range(len(encabezados))]
    alineacion = ["---" + "-" * (a - 3) if j == 0
                  else "-" * (a - 1) + ":" for j, a in enumerate(anchos)]

    def linea(campos):
        celdas_txt = [c.ljust(anchos[j]) if j == 0 else c.rjust(anchos[j])
                      for j, c in enumerate(campos)]
        return "| " + " | ".join(celdas_txt) + " |"

    partes = []
    if titulo:
        partes += [f"**{titulo}**", ""]
    partes += [linea(encabezados), linea(alineacion)]
    partes += [linea(f) for f in filas]
    for i, texto in enumerate(notas or []):
        partes += [""] if i == 0 else []
        partes += [f"{chr(97 + i)}. {texto}"]
    return "\n".join(partes) + "\n"


markdown = a_markdown(
    TABLA_INFORME,
    "Energía anual estimada de tres configuraciones del arreglo.",
    NOTAS)
(SALIDAS / "tablas" / "arreglo_fotovoltaico.md").write_text(
    markdown, encoding="utf-8")
print(markdown)

### 4.3 Las dos salidas dicen lo mismo

La comprobación que cierra la sección es la que importa. Si las
cifras de la versión en LaTeX y las de la versión en Markdown
coinciden, entonces ninguna transcripción manual se coló entre las
dos, que es el defecto contra el que el libro previene.

In [ ]:
import re


def numeros_de(texto: str) -> list[str]:
    # Extrae las cifras del texto, para comparar dos salidas.
    return re.findall(r"-?\d+\.?\d*", texto)


cuerpo_latex = latex[latex.index(r"\toprule"):latex.index(r"\bottomrule")]
cuerpo_markdown = "\n".join(
    l for l in markdown.splitlines()
    if l.startswith("|") and "---" not in l)

assert numeros_de(cuerpo_latex) == numeros_de(cuerpo_markdown)
print("Las dos salidas contienen exactamente las mismas cifras,")
print(f"{len(numeros_de(cuerpo_latex))} números en el mismo orden.")
print("\nArchivos escritos:")
for ruta in sorted((SALIDAS / "tablas").iterdir()):
    print(f"  {ruta.as_posix()}  ({ruta.stat().st_size} bytes)")

## 5. La comparación de escenarios que la tabla esconde

El Ejemplo 4.7 del libro advierte que comparar las tres
configuraciones por sus intervalos absolutos es incorrecto. La
diferencia entre la mejor y la de referencia vale 215 kWh frente a
márgenes de 460 kWh, de modo que parecerían indistinguibles. Sin
embargo la diferencia comparte la potencia y la razón de desempeño, y
las dos irradiaciones provienen del mismo modelo de radiación y están
correlacionadas con coeficiente cercano a 0.95.

La incertidumbre de la diferencia vale apenas 100 kWh, y la ventaja
alcanza poco más de dos incertidumbres típicas. Reportar solo los
intervalos absolutos oculta esa cancelación y lleva a declarar que no
hay diferencia donde sí la hay.

In [ ]:
CORRELACION_IRRADIACION = 0.95

base = resultados.loc["Base"]
optima = resultados.loc["Optima"]
diferencia = optima["E_kWh"] - base["E_kWh"]

u_h_base = base["u_rel_H"] * base["H_kWh_m2"]
u_h_optima = optima["u_rel_H"] * optima["H_kWh_m2"]
delta_h = optima["H_kWh_m2"] - base["H_kWh_m2"]

u_diferencia = np.sqrt(
    (RAZON_DESEMPENO * delta_h * U_POTENCIA)**2
    + (POTENCIA * delta_h * U_RAZON)**2
    + (POTENCIA * RAZON_DESEMPENO)**2
    * (u_h_base**2 + u_h_optima**2
       - 2 * CORRELACION_IRRADIACION * u_h_base * u_h_optima))

print(f"Diferencia entre la óptima y la base {diferencia:.1f} kWh   "
      f"(libro 215 kWh)")
print(f"Margen de cada configuración por separado "
      f"{base['u_E_kWh']:.0f} kWh   (libro 460 kWh)")
print(f"Incertidumbre de la diferencia {u_diferencia:.1f} kWh   "
      f"(libro cerca de 100 kWh)")
print(f"La ventaja alcanza {diferencia / u_diferencia:.2f} "
      f"incertidumbres típicas.")
assert abs(diferencia - 215.0) < 1.0
assert abs(u_diferencia - 100.0) < 1.0
assert diferencia / u_diferencia > 2.0

## 6. La figura de informe con la incertidumbre representada

La Figura 4.10 del libro exige que los ejes lleven símbolo, nombre y
unidad, que la incertidumbre se dibuje y que la leyenda no repita el
pie. La figura siguiente cumple las tres condiciones y añade la
comparación de la sección anterior, que es lo que el argumento
necesita y la tabla no transmite.

In [ ]:
def figura_de_informe():
    """Energía anual con su incertidumbre y la diferencia entre escenarios."""
    fig, (izq, der) = plt.subplots(
        1, 2, figsize=(10.0, 4.0),
        gridspec_kw={"width_ratios": [1.35, 1.0]})

    nombres = list(resultados.index)
    posiciones = np.arange(len(nombres))
    energia = resultados["E_kWh"].to_numpy() / 1000.0
    margen = resultados["u_E_kWh"].to_numpy() / 1000.0

    izq.bar(posiciones, energia, yerr=margen, capsize=5, width=0.58,
            color=[PALETA["azul"], PALETA["verde"], PALETA["naranja"]],
            edgecolor="white", error_kw={"ecolor": PALETA["gris"],
                                        "elinewidth": 1.2})
    for x_pos, valor, u_valor in zip(posiciones, energia, margen):
        izq.text(x_pos, valor + u_valor + 0.25,
                 f"{valor:.2f} +- {u_valor:.2f}",
                 ha="center", fontsize=8, color=PALETA["gris"])
    izq.set_xticks(posiciones)
    izq.set_xticklabels(nombres)
    izq.set_ylabel("Energía anual E (MWh)")
    izq.set_xlabel("Configuración del arreglo")
    izq.set_ylim(0.0, 9.6)
    izq.set_title("(a) energía con incertidumbre absoluta")

    etiquetas = ["Óptima menos Base", "Sombreado menos Base"]
    deltas = np.array([
        resultados.loc["Optima", "E_kWh"] - resultados.loc["Base", "E_kWh"],
        resultados.loc["Sombreado", "E_kWh"] - resultados.loc["Base", "E_kWh"]])
    u_deltas = np.array([u_diferencia, u_diferencia])
    der.errorbar(deltas, np.arange(len(deltas)), xerr=u_deltas, fmt="o",
                 color=PALETA["rojo"], capsize=5, ms=6,
                 label="diferencia con su incertidumbre")
    der.errorbar(deltas, np.arange(len(deltas)) + 0.28,
                 xerr=np.sqrt(2) * resultados["u_E_kWh"].iloc[0],
                 fmt="s", color=PALETA["gris"], capsize=5, ms=5,
                 alpha=0.8, label="si se ignorara la correlación")
    der.axvline(0.0, color=PALETA["gris"], lw=0.9)
    der.set_yticks(np.arange(len(deltas)) + 0.14)
    der.set_yticklabels(etiquetas)
    der.set_xlabel("Diferencia de energía anual (kWh)")
    der.set_title("(b) la comparación que la tabla esconde")
    der.legend(loc="lower right", fontsize=7.5)

    fig.tight_layout()
    return fig


figura = figura_de_informe()
plt.show()

### 6.1 Exportación en PDF vectorial

El formato vectorial es el que el documento final necesita, porque
conserva el texto como texto y la línea como línea, de modo que
escala sin pérdida y permite buscar dentro de la figura. Exportar en
mapa de bits es el error que más se paga en la imprenta.

La comprobación siguiente abre el archivo producido y verifica que
contenga descripciones de tipografía y ningún objeto de imagen, que
es la firma de un PDF verdaderamente vectorial.

In [ ]:
import re as _re


def es_pdf_vectorial(ruta) -> tuple[bool, bool, bool]:
    # Devuelve si abre como PDF, si trae tipografía y si trae un mapa
    # de bits incrustado. La marca de una imagen rasterizada es un
    # objeto con subtipo Image. Las entradas ImageB, ImageC e ImageI
    # del conjunto de procedimientos aparecen siempre y no cuentan.
    bytes_pdf = Path(ruta).read_bytes()
    es_pdf = bytes_pdf.startswith(b"%PDF")
    tipografia = _re.search(rb"/Type\s*/Font", bytes_pdf) is not None
    rasterizado = _re.search(rb"/Subtype\s*/Image", bytes_pdf) is not None
    return es_pdf, tipografia, rasterizado


RUTA_PDF = SALIDAS / "figuras" / "energia_anual.pdf"
figura.savefig(RUTA_PDF, format="pdf", bbox_inches="tight")
figura.savefig(SALIDAS / "figuras" / "energia_anual.png", dpi=200)
plt.close(figura)

es_pdf, tiene_tipografia, tiene_imagen = es_pdf_vectorial(RUTA_PDF)
tamano_pdf = RUTA_PDF.stat().st_size
tamano_png = (SALIDAS / "figuras" / "energia_anual.png").stat().st_size

print(f"Archivo {RUTA_PDF.as_posix()}, {tamano_pdf} bytes")
print(f"Abre como PDF: {es_pdf}")
print(f"Contiene descripciones de tipografía: {tiene_tipografia}")
print(f"Contiene una imagen rasterizada incrustada: {tiene_imagen}")
print(f"La versión en mapa de bits pesa {tamano_png} bytes, "
      f"{tamano_png / tamano_pdf:.1f} veces más")
assert es_pdf and tiene_tipografia and not tiene_imagen
print("\nEl PDF es vectorial, que es lo que el documento final necesita.")

### 6.2 El pie de figura que interpreta

El libro insiste en que el pie interprete en lugar de describir. Un
pie que dice lo que la figura es resulta inútil, porque el lector ya
lo ve; un pie que dice lo que la figura permite concluir es el que
sostiene el argumento. El cuaderno lo escribe junto a la figura para
que viaje con ella.

In [ ]:
PIE_FIGURA = (
    "Energía anual estimada de las tres configuraciones del arreglo "
    "fotovoltaico de 5.00 kW, con su incertidumbre típica. Los "
    "intervalos absolutos de la izquierda se traslapan, mientras que "
    "la diferencia de la derecha, que cancela la incertidumbre común "
    "de la potencia, de la razón de desempeño y del modelo de "
    "radiación, separa la configuración óptima de la base por más de "
    "dos incertidumbres típicas.")

(SALIDAS / "figuras" / "energia_anual.txt").write_text(
    PIE_FIGURA, encoding="utf-8")
print(PIE_FIGURA)
assert len(PIE_FIGURA.split()) > 40

## 7. Ejercicios guiados

Las celdas siguientes llevan la marca `# COMPLETE:` y arrancan con un
valor de partida evidentemente incorrecto, de modo que el cuaderno
corre completo aunque falten por resolver. Al terminarlas, cambie
`REVISAR = True` en la celda de configuración.

### Ejercicio 1. Decimales que la incertidumbre gobierna

Complete la función que decide cuántos decimales lleva una columna a
partir de las incertidumbres que aparecen en ella. La función
`posicion_significativa` ya está definida en la sección 3.

In [ ]:
def decimales_de_columna(incertidumbres, cifras=2):
    """Decimales que hacen coherente una columna de resultados."""
    # COMPLETE: devuelva el mayor de los valores que
    # posicion_significativa entrega para cada incertidumbre, con un
    # mínimo de cero.
    return None


decimales_energia = decimales_de_columna([0.455, 0.468, 0.439])
decimales_caudal = decimales_de_columna([1.518e-3])

In [ ]:
# Verificación del ejercicio 1.
verificar("decimales de la columna de energía", decimales_energia, 2.0)
verificar("decimales de una columna en m3/s", decimales_caudal, 4.0)
if decimales_caudal is not None:
    print(f"\nUna incertidumbre de 1.5e-3 m3/s pide {decimales_caudal} "
          f"decimales, y expresar el caudal en L/s los baja a uno.")

### Ejercicio 2. Rendimiento específico con su incertidumbre

El rendimiento específico es la energía dividida por la potencia
nominal, de modo que vale la irradiación por la razón de desempeño.
Calcule su incertidumbre relativa, recordando que la potencia se
cancela por aparecer en el numerador de la energía y en el
denominador del rendimiento.

In [ ]:
# COMPLETE: combine solo las dos fuentes que quedan, la irradiación y
# la razón de desempeño, y guarde el resultado en u_rel_rendimiento.
u_rel_rendimiento = None

In [ ]:
# Verificación del ejercicio 2.
verificar("incertidumbre relativa del rendimiento", u_rel_rendimiento,
          0.0554914, tol=1.0e-4)
if u_rel_rendimiento is not None:
    print(f"\nRendimiento {100 * u_rel_rendimiento:.2f} por ciento frente a")
    print(f"energía {100 * u_relativa.iloc[0]:.2f} por ciento. La potencia se")
    print("cancela porque entra en el numerador y en el denominador.")

### Ejercicio 3. Una tabla de informe completa

Arme la tabla del rendimiento específico con su incertidumbre, con la
unidad declarada una sola vez en el encabezado y los decimales que la
incertidumbre gobierne. Use `tabla_de_informe`.

In [ ]:
rendimiento = resultados["Y_kWh_kW"].to_numpy()
u_rendimiento = (u_rel_rendimiento or 0.0) * rendimiento

# COMPLETE: construya TABLA_RENDIMIENTO con tabla_de_informe, con las
# columnas Y y u_Y, encabezados "Y (kWh/kW)" y "u(Y) (kWh/kW)", y el
# número de decimales que decimales_de_columna entregue.
TABLA_RENDIMIENTO = None

In [ ]:
# Verificación del ejercicio 3.
if TABLA_RENDIMIENTO is None:
    print("[pendiente] la celda marcada COMPLETE sigue sin resolver")
else:
    print(TABLA_RENDIMIENTO.to_string())
    correcta = (list(TABLA_RENDIMIENTO.columns)
                == ["Y (kWh/kW)", "u(Y) (kWh/kW)"]
                and TABLA_RENDIMIENTO.iloc[0, 0] == "1544")
    print(f"[{'ok' if correcta else 'revisar'}] encabezados y redondeo")
    if REVISAR:
        assert correcta

### Ejercicio 4. La misma tabla en los dos formatos

Exporte la tabla del rendimiento a LaTeX y a Markdown y compruebe que
sus cifras coinciden, que es la comprobación que impide que se cuele
una transcripción manual entre las dos.

In [ ]:
# COMPLETE: genere las dos salidas de TABLA_RENDIMIENTO con a_latex y
# a_markdown, y guarde en cifras_iguales el resultado de comparar sus
# listas de números con la función numeros_de.
cifras_iguales = None

In [ ]:
# Verificación del ejercicio 4.
if cifras_iguales is None:
    print("[pendiente] la celda marcada COMPLETE sigue sin resolver")
else:
    print(f"[{'ok' if cifras_iguales else 'revisar'}] las dos salidas "
          f"contienen las mismas cifras")
    if REVISAR:
        assert cifras_iguales

### Ejercicio 5. Una figura con la incertidumbre representada

Dibuje el rendimiento específico de las tres configuraciones con su
barra de incertidumbre, expórtelo en PDF vectorial y verifique el
archivo con la misma comprobación de la sección 6.1.

In [ ]:
# COMPLETE: construya la figura con plt.subplots, dibuje una barra de
# error por configuración con errorbar, rotule los ejes con nombre y
# unidad, guarde en SALIDAS/figuras/rendimiento.pdf y deje en
# ruta_rendimiento la ruta del archivo escrito.
ruta_rendimiento = None

In [ ]:
# Verificación del ejercicio 5.
if ruta_rendimiento is None:
    print("[pendiente] la celda marcada COMPLETE sigue sin resolver")
else:
    es_pdf_r, tipografia_r, raster_r = es_pdf_vectorial(ruta_rendimiento)
    vectorial = es_pdf_r and tipografia_r and not raster_r
    print(f"[{'ok' if vectorial else 'revisar'}] "
          f"{Path(ruta_rendimiento).as_posix()} es PDF vectorial, "
          f"{Path(ruta_rendimiento).stat().st_size} bytes")
    if REVISAR:
        assert vectorial

## 8. Problemas del capítulo

### Problema 4-28, resuelto

Escriba un guion que regenere todas las figuras y tablas del informe
desde los datos crudos, con semilla declarada y sin rutas absolutas.
La celda siguiente lo escribe en `salidas/reconstruir_tablas.py` y lo
deja listo para ejecutarse desde la raíz del proyecto.

In [ ]:
LINEAS_GUION = [
    "# Regenera las tablas del informe desde los datos crudos.",
    "# Uso, desde la raíz del proyecto:",
    "#     python3 salidas/reconstruir_tablas.py",
    "from pathlib import Path",
    "",
    "import matplotlib",
    "matplotlib.use('Agg')",
    "import numpy as np",
    "import pandas as pd",
    "",
    "SEMILLA = 20262",
    "RAIZ = Path(__file__).resolve().parent.parent",
    "DATOS = RAIZ / 'datos'",
    "SALIDAS = Path(__file__).resolve().parent",
    "POTENCIA, U_POTENCIA = 5.00, 0.10",
    "RAZON_DESEMPENO, U_RAZON = 0.78, 0.03",
    "",
    "",
    "def main() -> None:",
    "    arreglo = pd.read_csv(DATOS / 'arreglo_fotovoltaico.csv')",
    "    marco = arreglo.set_index('configuracion')",
    "    u_rel = np.sqrt((U_POTENCIA / POTENCIA) ** 2",
    "                    + marco['u_rel_H'] ** 2",
    "                    + (U_RAZON / RAZON_DESEMPENO) ** 2)",
    "    marco['E_kWh'] = POTENCIA * marco['H_kWh_m2'] * RAZON_DESEMPENO",
    "    marco['u_E_kWh'] = u_rel * marco['E_kWh']",
    "    (SALIDAS / 'tablas').mkdir(parents=True, exist_ok=True)",
    "    marco.to_csv(SALIDAS / 'tablas' / 'energia_anual.csv')",
    "    print('tabla regenerada con semilla', SEMILLA)",
    "",
    "",
    "if __name__ == '__main__':",
    "    main()",
]

ruta_guion = SALIDAS / "reconstruir_tablas.py"
ruta_guion.write_text("\n".join(LINEAS_GUION) + "\n", encoding="utf-8")
texto_guion = ruta_guion.read_text(encoding="utf-8")

print(f"Guion escrito en {ruta_guion.as_posix()}, "
      f"{ruta_guion.stat().st_size} bytes")
compile(texto_guion, str(ruta_guion), "exec")
# Una ruta absoluta se delata porque una cadena empieza por la raíz
# del sistema de archivos o por una unidad de disco.
RAICES_ABSOLUTAS = (chr(34) + "/", chr(39) + "/", ":" + chr(92))

assert "SEMILLA = 20262" in texto_guion
assert not any(r in texto_guion for r in RAICES_ABSOLUTAS)
print("El guion compila, fija la semilla y no declara ninguna ruta")
print("absoluta, que son las tres condiciones del problema 4-28.")

### Problema 4-31, andamiaje

Un cliente exige una predicción puntual y rechaza el intervalo.
Redacte la respuesta profesional que lo defiende y la forma de
presentarlo que él sí aceptaría. La celda siguiente construye esa
forma alternativa, que convierte el intervalo en una afirmación de
riesgo con la que un cliente sabe decidir.

In [ ]:
energia_base = resultados.loc["Base", "E_kWh"]
u_base = resultados.loc["Base", "u_E_kWh"]
generador_riesgo = np.random.default_rng(SEMILLA)
escenarios = generador_riesgo.normal(energia_base, u_base, 100_000)

for umbral in (7000.0, 7500.0, 8000.0):
    probabilidad = 100 * np.mean(escenarios >= umbral)
    print(f"Probabilidad de superar {umbral:.0f} kWh al año: "
          f"{probabilidad:.1f} por ciento")

percentil_conservador = np.percentile(escenarios, 10)
print(f"\nValor que se supera nueve veces de cada diez: "
      f"{percentil_conservador:.0f} kWh")
print("Esa es la cifra puntual que el cliente puede usar sin engañarse,")
print("porque lleva escrito el nivel de confianza con que se cumple.")
assert percentil_conservador < energia_base

## Cierre

### Lo que debe saber hacer al terminar

- Decidir si un resultado pide tabla o figura, con el criterio de función y no de costumbre.
- Generar la tabla de resultados desde el DataFrame del proyecto, sin transcribir ninguna cifra.
- Redondear valor e incertidumbre a la misma posición decimal de manera automática, columna por columna.
- Exportar la misma tabla a LaTeX y a Markdown desde una sola fuente, y comprobar que ambas coinciden.
- Dibujar la incertidumbre en la figura de informe y exportarla en PDF vectorial, verificando que el texto quedó como texto.

### Qué revisar en el libro si algo no salió

- Si duda de qué lleva una tabla y qué lleva una figura, la Figura 4.10 del libro señala cada elemento.
- Si el redondeo no le cuadra, la sección 4.4.2 enuncia la regla de las cifras significativas.
- Si la comparación entre escenarios le parece contradictoria, el Ejemplo 4.7 explica la cancelación de la incertidumbre común.
- Si el informe no se regenera con un solo comando, revise la Definición 4.11 y la Figura 4.11.

### Declaración del uso de asistentes de programación

Este cuaderno se preparó con apoyo de un asistente de programación.
Todo resultado numérico que aparece aquí se verifica contra el valor
que el libro publica, contra una solución analítica o contra un caso
límite, según recuerda la sección 4.6 del libro. La responsabilidad
del contenido no se transfiere al asistente.